# JLS seuils heuristiques
Sweep des seuils margin / confidence et export des metriques.

In [5]:
import ast
import json
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import f1_score, accuracy_score
from roles_parser import parse_all_clauses

# --- IO ---
base_dir = Path.cwd()
out_dir = base_dir / "analyse" / "csv_heuristique_study" / "threshold_sweep"
out_dir.mkdir(parents=True, exist_ok=True)

# --- helpers ---
def _parse_tokens(value):
    if value is None or str(value).strip() == "":
        return []
    txt = str(value).strip()
    try:
        return json.loads(txt)
    except Exception:
        try:
            return ast.literal_eval(txt)
        except Exception:
            return []

def _parse_dict(value):
    if value is None or str(value).strip() == "":
        return None
    txt = str(value).strip()
    try:
        return json.loads(txt)
    except Exception:
        try:
            return ast.literal_eval(txt)
        except Exception:
            return None

def _candidate_conf(candidates, source, default=0.0):
    for item in candidates or []:
        if item.get("source") == source:
            return item.get("conf", default)
    return default

def _boundary_strength(debug):
    strength = debug.get("boundary_reset")
    return strength if strength in {"strong", "weak"} else None

def _coverage_table(df, rule_col, needs_col, out_csv):
    if rule_col not in df.columns:
        print(f"Missing column: {rule_col}")
        return
    g = df.groupby(rule_col, dropna=False)[needs_col].value_counts(dropna=False).unstack(fill_value=0)
    g = g.reindex(columns=[False, True], fill_value=0)
    g.columns = ["needs_ml_false", "needs_ml_true"]
    g["N"] = g["needs_ml_false"] + g["needs_ml_true"]
    g = g.reset_index().rename(columns={rule_col: "rule"}).sort_values("N", ascending=False)
    rare = g["N"] < 10
    if rare.any():
        other = pd.DataFrame({
            "rule": ["OTHER(<10)"],
            "needs_ml_false": [g.loc[rare, "needs_ml_false"].sum()],
            "needs_ml_true": [g.loc[rare, "needs_ml_true"].sum()],
            "N": [g.loc[rare, "N"].sum()],
        })
        g = pd.concat([g.loc[~rare], other], ignore_index=True)
    total = g["N"].sum()
    g["pct_total"] = (g["N"] / total * 100).round(2)
    g["pct_needs_ml_true"] = (g["needs_ml_true"] / g["N"] * 100).round(2)
    g.to_csv(out_csv, index=False)

def _safe_metrics(y_true, y_pred):
    if len(y_true) == 0:
        return np.nan, np.nan
    labels = sorted(pd.unique(y_true))
    macro_f1 = f1_score(y_true, y_pred, average="macro", labels=labels, zero_division=0)
    acc = accuracy_score(y_true, y_pred)
    return macro_f1, acc

# --- load clause_json and parse ---
df_clauses = pd.read_csv("data/clause_json.csv")

clauses = []
for _, row in df_clauses.iterrows():
    clause_flags = _parse_dict(row.get("clause_flags")) or {}
    boundary_strength = row.get("boundary_strength")
    if isinstance(boundary_strength, str) and boundary_strength.strip():
        clause_flags["boundary_strength"] = boundary_strength.strip()

    clauses.append({
        "tokens": _parse_tokens(row.get("tokens")),
        "split": _parse_dict(row.get("split")),
        "boundary_seen": bool(row.get("boundary_seen")),
        "clause_flags": clause_flags,
        "clause_index": row.get("clause_index"),
        "sentence_id": row.get("sentence_id"),
        "file": row.get("file"),
        "tier": row.get("tier"),
        "start": row.get("start"),
        "end": row.get("end"),
    })

parsed = parse_all_clauses(clauses)

rows = []
for clause_id, clause in enumerate(parsed):
    action = clause.get("action", {})
    agent = clause.get("agent", {})
    patient = clause.get("patient", {})
    debug = clause.get("debug", {})
    candidates = debug.get("agent_candidates") or []
    rows.append({
        "clause_id": clause_id,
        "action_value": action.get("value"),
        "action_conf": action.get("conf"),
        "action_rule": action.get("rule"),
        "agent_value": agent.get("value"),
        "agent_conf": agent.get("conf"),
        "agent_rule": agent.get("rule"),
        "patient_value": patient.get("value"),
        "patient_conf": patient.get("conf"),
        "patient_rule": patient.get("rule"),
        "needs_ml_agent": debug.get("needs_ml_agent"),
        "needs_ml_patient": debug.get("needs_ml_patient"),
        "agent_candidates": debug.get("agent_candidates"),
        "patient_candidates": debug.get("patient_candidates"),
        "topic_conf": _candidate_conf(candidates, "TOPIC", 0.0),
        "last_agent_conf": _candidate_conf(candidates, "LAST_AGENT", 0.0),
        "last_patient_conf": _candidate_conf(candidates, "LAST_PATIENT", 0.0),
        "unknown_conf": _candidate_conf(candidates, "UNKNOWN", 0.10),
        "has_verb": debug.get("verb_index", -1) != -1,
        "verb_index": debug.get("verb_index"),
        "noun_count": debug.get("noun_count"),
        "postverb_is_pt": debug.get("postverb_is_pt"),
        "postverb_pt_resolved": debug.get("pt_resolved"),
        "pt_is_bare": debug.get("r13_reason") == "PT_BARE_NONBLOCKING",
        "boundary_strength": _boundary_strength(debug),
    })

df_parsed = pd.DataFrame(rows)

# discourse stats
df_parsed["discourse_max"] = df_parsed[["topic_conf", "last_agent_conf", "last_patient_conf"]].max(axis=1)
df_parsed["discourse_second"] = df_parsed[["topic_conf", "last_agent_conf", "last_patient_conf"]].apply(
    lambda r: r.nlargest(2).iloc[-1], axis=1
)
df_parsed["discourse_margin"] = df_parsed["discourse_max"] - df_parsed["discourse_second"]

# trivial_unknown filter
trivial_unknown = (
    (df_parsed["topic_conf"] == 0)
    & (df_parsed["last_agent_conf"] == 0)
    & (df_parsed["last_patient_conf"] == 0)
)
df_parsed.loc[trivial_unknown, "needs_ml_agent"] = False
df_parsed.loc[trivial_unknown, "needs_ml_patient"] = False

# --- annotated for metrics (optional) ---
df_lab = pd.read_csv("clauses_labels_train.csv") if Path("clauses_labels_train.csv").exists() else None

# --- sweep ---
t_values = [0.03, 0.07, 0.12, 0.17, 0.22, 0.27, 0.32]
c_values = [0.45, 0.55, 0.65, 0.70, 0.75, 0.80, 0.85]


rows_summary = []
for t in t_values:
    for c in c_values:
        df_run = df_parsed.copy()
        needs_ml = (df_run["discourse_margin"] < t) | (df_run["discourse_max"] < c)
        df_run["needs_ml_agent_thr"] = needs_ml
        df_run["needs_ml_patient_thr"] = needs_ml

        # UNKNOWN override
        df_run["agent_pred_thr"] = df_run["agent_value"].astype(str)
        df_run["patient_pred_thr"] = df_run["patient_value"].astype(str)
        df_run.loc[needs_ml, "agent_pred_thr"] = "UNKNOWN"
        df_run.loc[needs_ml, "patient_pred_thr"] = "UNKNOWN"

        # coverage tables
        _coverage_table(df_run, "agent_rule", "needs_ml_agent_thr", out_dir / f"coverage_agent_rules_t{t}_c{c}.csv")
        _coverage_table(df_run, "patient_rule", "needs_ml_patient_thr", out_dir / f"coverage_patient_rules_t{t}_c{c}.csv")

        # metrics on annotated
        macro_f1_agent = np.nan
        acc_agent = np.nan
        macro_f1_patient = np.nan
        acc_patient = np.nan

        if df_lab is not None and "clause_id" in df_lab.columns:
            df_eval = df_lab.merge(df_run, on="clause_id", how="inner")
            if "agent label" in df_eval.columns:
                y_true = df_eval["agent label"].astype(str)
                y_pred = df_eval["agent_pred_thr"].astype(str)
                macro_f1_agent, acc_agent = _safe_metrics(y_true, y_pred)
            if "patient labels" in df_eval.columns:
                y_true = df_eval["patient labels"].astype(str)
                y_pred = df_eval["patient_pred_thr"].astype(str)
                macro_f1_patient, acc_patient = _safe_metrics(y_true, y_pred)

        rows_summary.append({
            "t_margin": t,
            "c_conf": c,
            "pct_needs_ml_agent": round(df_run["needs_ml_agent_thr"].mean() * 100, 2),
            "pct_needs_ml_patient": round(df_run["needs_ml_patient_thr"].mean() * 100, 2),
            "pct_unknown_agent": round((df_run["agent_pred_thr"] == "UNKNOWN").mean() * 100, 2),
            "pct_unknown_patient": round((df_run["patient_pred_thr"] == "UNKNOWN").mean() * 100, 2),
            "macro_f1_agent": macro_f1_agent,
            "acc_agent": acc_agent,
            "macro_f1_patient": macro_f1_patient,
            "acc_patient": acc_patient,
        })

summary = pd.DataFrame(rows_summary)
summary.to_csv(out_dir / "threshold_sweep_summary.csv", index=False)
print("Saved:", out_dir / "threshold_sweep_summary.csv")
print("Coverage tables saved per (t,c) in:", out_dir)


Saved: c:\Users\Ewan\Desktop\Projet JAPON JSL\python project\analyse\csv_heuristique_study\threshold_sweep\threshold_sweep_summary.csv
Coverage tables saved per (t,c) in: c:\Users\Ewan\Desktop\Projet JAPON JSL\python project\analyse\csv_heuristique_study\threshold_sweep


In [6]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

out_dir = Path("analyse") / "csv_heuristique_study" / "threshold_sweep"
out_dir.mkdir(parents=True, exist_ok=True)

summary_path = out_dir / "threshold_sweep_summary.csv"
if not summary_path.exists():
    print("Missing:", summary_path)
else:
    df = pd.read_csv(summary_path)

    # Heatmaps macro-F1
    for metric, title in [
        ("macro_f1_agent", "Macro-F1 agent"),
        ("macro_f1_patient", "Macro-F1 patient"),
        ("pct_needs_ml_agent", "% needs_ml agent"),
        ("pct_needs_ml_patient", "% needs_ml patient"),
    ]:
        pivot = df.pivot(index="t_margin", columns="c_conf", values=metric)
        plt.figure(figsize=(6, 4))
        sns.heatmap(pivot, annot=True, fmt=".2f", cmap="viridis")
        plt.title(title)
        plt.xlabel("c_conf")
        plt.ylabel("t_margin")
        plt.tight_layout()
        plt.savefig(out_dir / f"heatmap_{metric}.png", dpi=150)
        plt.close()

    # Curves: % needs_ml vs t for each c
    for metric, title in [
        ("pct_needs_ml_agent", "% needs_ml agent"),
        ("pct_needs_ml_patient", "% needs_ml patient"),
    ]:
        plt.figure(figsize=(6, 4))
        for c in sorted(df["c_conf"].unique()):
            sub = df[df["c_conf"] == c].sort_values("t_margin")
            plt.plot(sub["t_margin"], sub[metric], marker="o", label=f"c={c}")
        plt.title(title)
        plt.xlabel("t_margin")
        plt.ylabel(metric)
        plt.legend(loc="best")
        plt.tight_layout()
        plt.savefig(out_dir / f"curve_{metric}.png", dpi=150)
        plt.close()

    print("Saved heatmaps + curves in:", out_dir)


Saved heatmaps + curves in: analyse\csv_heuristique_study\threshold_sweep
